<a href="https://colab.research.google.com/github/anjalii-s/Multi-Agent-Loan-Underwriting-Assistant_-CREW-AI/blob/main/CreditGuard_AI_CrewAI_Groq_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CreditGuard AI — Multi-Agent Loan Underwriting Assistant

**A CrewAI showcase project (runs free on Groq + Google Colab)**

### The real-world problem
At small lenders and fintechs, a single loan application still passes through several manual steps before a human underwriter signs off:
1. A **financial analyst** checks income vs. debt (DTI ratio) and credit history.
2. A **risk officer** turns those numbers into a risk tier.
3. A **compliance officer** re-reads the file for KYC/AML red flags (unverifiable address, mismatched documents, etc.).
4. Someone writes up a **memo** so the final decision-maker doesn't have to re-derive all of the above.

This is slow, inconsistent between reviewers, and hard to audit. This notebook builds a **4-agent CrewAI crew** that performs the same first-pass screening in seconds, with each agent playing one real underwriting role — and it stays fully explainable, since every number and flag is produced by a tool, not guessed by the LLM.

> This mirrors the actual pre-screening workflow used in retail/consumer lending, and is built for a portfolio, not for production lending decisions.

### Stack
- **CrewAI** — multi-agent orchestration (roles, tasks, sequential process)
- **Groq** — free, extremely fast LLM inference (Llama 3.3 70B)
- **Google Colab (free tier)** — no local setup, no GPU needed (Groq does the inference)

### Get a free Groq API key
https://console.groq.com/keys — sign up, create a key, paste it when prompted below.

## 1. Install dependencies

In [27]:
!pip install -q crewai crewai-tools litellm

## 2. Set  Groq API key
Get a free key at https://console.groq.com/keys — it's not stored anywhere except this Colab session.

In [28]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    os.environ["GROQ_API_KEY"] = getpass("Paste your free Groq API key: ")


## 3. Sample applications
Synthetic data only — no real names, no real PII.

In [29]:
loan_applications = [
    {
        "applicant_id": "APP-1001",
        "name": "Applicant A",
        "monthly_income": 3200,
        "monthly_debt_payments": 1450,
        "requested_loan_amount": 15000,
        "loan_purpose": "Debt consolidation",
        "employment_years": 2.5,
        "credit_score": 612,
        "country": "Latvia",
        "flags_raw_text": "Applicant listed a P.O. box as home address and could not confirm employer phone number.",
    },
    {
        "applicant_id": "APP-1002",
        "name": "Applicant B",
        "monthly_income": 5400,
        "monthly_debt_payments": 900,
        "requested_loan_amount": 8000,
        "loan_purpose": "Home renovation",
        "employment_years": 7,
        "credit_score": 748,
        "country": "Estonia",
        "flags_raw_text": "No inconsistencies noted. Documents match employer records.",
    },
]


## 4. Tools

These are plain Python functions wrapped as CrewAI tools. This is deliberate: **the math and the KYC keyword screen never touch the LLM** — the LLM only reasons over tool outputs and writes the narrative. That keeps the numbers auditable, which matters a lot in a credit-risk context.

In [30]:
from crewai.tools import tool

@tool("DTI Calculator")
def calculate_dti(monthly_income: float, monthly_debt_payments: float) -> str:
    """Calculates the Debt-to-Income (DTI) ratio given monthly income and existing
    monthly debt payments. Returns the DTI as a percentage and a risk band
    (Low <36%, Medium 36-43%, High >43%)."""
    if monthly_income <= 0:
        return "Invalid income: cannot compute DTI."
    dti = (monthly_debt_payments / monthly_income) * 100
    if dti < 36:
        band = "Low"
    elif dti <= 43:
        band = "Medium"
    else:
        band = "High"
    return f"DTI = {dti:.1f}% -> Risk band: {band}"


WATCHLIST_PHRASES = ["p.o. box", "could not confirm", "mismatch", "unverified", "refused to provide"]

@tool("KYC Red Flag Screener")
def screen_kyc_flags(notes_text: str) -> str:
    """Scans free-text underwriting notes for common KYC/AML red-flag phrases
    (e.g. unverifiable address, mismatched documents) and returns any matches found."""
    text = notes_text.lower()
    hits = [p for p in WATCHLIST_PHRASES if p in text]
    if hits:
        return f"Red flags detected: {hits}"
    return "No red-flag phrases detected in notes."


@tool("Credit Score Bracket")
def credit_score_bracket(score: int) -> str:
    """Maps a numeric credit score (300-850 scale) to a qualitative bracket:
    Excellent (740+), Good (670-739), Fair (580-669), Poor (<580)."""
    if score >= 740:
        return "Excellent"
    elif score >= 670:
        return "Good"
    elif score >= 580:
        return "Fair"
    else:
        return "Poor"


## 5. Agents

Four agents, four distinct underwriting roles. Each `role`/`goal`/`backstory` is written to keep the agent narrowly scoped — this is what stops a single LLM call from vaguely doing "everything at once" and instead produces a traceable division of labor, the same way a real credit team is organized.

In [31]:
from crewai import Agent, Task, Crew, Process, LLM

groq_llm = LLM(
    model="groq/llama-3.3-70b-versatile",  # fast + generous free tier
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2,
)

financial_analyst = Agent(
    role="Financial Analyst",
    goal="Extract and compute the key financial ratios needed to assess a loan applicant's repayment capacity.",
    backstory=("You are a meticulous credit operations analyst with years of experience reviewing "
               "retail loan files at a commercial bank. You never skip a ratio calculation."),
    tools=[calculate_dti, credit_score_bracket],
    llm=groq_llm,
    verbose=True,
)

risk_assessor = Agent(
    role="Credit Risk Assessor",
    goal="Translate financial facts into a clear risk tier (Low/Medium/High) with a short justification.",
    backstory=("You are a senior credit risk officer who has underwritten thousands of consumer loans. "
               "You weigh income stability, DTI, and credit history together, never in isolation."),
    llm=groq_llm,
    verbose=True,
)

compliance_officer = Agent(
    role="Compliance Officer",
    goal="Screen the application notes for KYC/AML red flags and state whether the file can proceed.",
    backstory=("You are a bank compliance officer trained in KYC/AML procedures. You are cautious "
               "and flag anything unverifiable, but you don't block files without cause."),
    tools=[screen_kyc_flags],
    llm=groq_llm,
    verbose=True,
)

report_writer = Agent(
    role="Underwriting Report Writer",
    goal=("Combine the financial, risk, and compliance findings into a short, professional "
          "underwriting memo with a final recommendation."),
    backstory=("You write the final memo that a human underwriter signs off on. You are concise, "
               "structured, and never invent information you were not given."),
    llm=groq_llm,
    verbose=True,
)


## 6. Tasks + Crew

`context=[...]` on a `Task` is what lets the Risk Assessor see the Financial Analyst's output, and lets the Report Writer see all three upstream results — this is CrewAI's way of chaining agent outputs without  manually passing strings around.

In [32]:
def build_crew(app: dict) -> Crew:
    financial_task = Task(
        description=(
            f"Applicant {app['applicant_id']} reports monthly income of {app['monthly_income']} "
            f"and monthly debt payments of {app['monthly_debt_payments']}. Their credit score is "
            f"{app['credit_score']}. Calculate the DTI ratio and the credit score bracket using "
            f"your tools, and summarize both in 2-3 sentences."
        ),
        expected_output="A short summary stating the DTI %, its risk band, and the credit score bracket.",
        agent=financial_analyst,
    )

    compliance_task = Task(
        description=(
            f"Review these underwriting notes for applicant {app['applicant_id']}: "
            f"\"{app['flags_raw_text']}\". Use your screening tool to check for red flags and "
            f"state clearly whether the file has any KYC concerns."
        ),
        expected_output="A short statement of any red flags found (or none) and a proceed/hold recommendation.",
        agent=compliance_officer,
    )

    risk_task = Task(
        description=(
            f"Using the financial analyst's findings, assess the overall credit risk tier "
            f"(Low/Medium/High) for applicant {app['applicant_id']}, who has {app['employment_years']} "
            f"years at their current employer and is requesting a loan of {app['requested_loan_amount']} "
            f"for '{app['loan_purpose']}'. Justify the tier in 2-3 sentences."
        ),
        expected_output="A stated risk tier (Low/Medium/High) with a brief justification.",
        agent=risk_assessor,
        context=[financial_task],
    )

    report_task = Task(
        description=(
            f"Write a short underwriting memo for applicant {app['applicant_id']} ({app['name']}) "
            f"combining the financial summary, the risk tier, and the compliance finding. "
            f"End with a final recommendation: Approve, Refer for manual review, or Decline."
        ),
        expected_output=(
            "A structured memo with sections: Applicant, Financial Summary, Risk Tier, "
            "Compliance Check, Final Recommendation."
        ),
        agent=report_writer,
        context=[financial_task, risk_task, compliance_task],
    )

    return Crew(
        agents=[financial_analyst, risk_assessor, compliance_officer, report_writer],
        tasks=[financial_task, compliance_task, risk_task, report_task],
        process=Process.sequential,
        verbose=True,
    )


In [35]:

import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [36]:
import os
from getpass import getpass


key = getpass("Paste your free Groq API key (starts with 'gsk_'): ").strip()

if not key.startswith("gsk_"):
    print("⚠️ That doesn't look like a Groq key — Groq keys start with 'gsk_'. Double-check you copied it fully.")

os.environ["GROQ_API_KEY"] = key

Paste your free Groq API key (starts with 'gsk_'): ··········


## 7. Run the crew on every sample application

In [37]:
results = {}
for app in loan_applications:
    print(f"\n{'='*60}\nProcessing {app['applicant_id']} — {app['name']}\n{'='*60}")
    crew = build_crew(app)
    output = await crew.kickoff_async()
    results[app["applicant_id"]] = output
    print(f"\n--- FINAL MEMO: {app['applicant_id']} ---\n{output}\n")


Processing APP-1001 — Applicant A


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0cc0eb0a-5fb9-47b9-930f-67ddb0d7f824                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant APP-1001 reports monthly income of 3200 and monthly debt payments of 1450. Their credit score  │
│  is 612. Calculate the DTI ratio and the credit score bracket using your tools, and summarize both in 2-3       │
│  sentences.                                                                                                     │
│  ID: 76d0a87e-7a79-45bc-833e-fcfe231040b7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Analyst                                                                                       │
│                                                                                                                 │
│  Task: Applicant APP-1001 reports monthly income of 3200 and monthly debt payments of 1450. Their credit score  │
│  is 612. Calculate the DTI ratio and the credit score bracket using your tools, and summarize both in 2-3       │
│  sentences.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: dti_calculator                                                                                           │
│  Args: {'monthly_debt_payments': 1450, 'monthly_income': 3200}                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Underwriting Report Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Underwriting Memo for Applicant APP-1002                                                                    │
│                                                                                                                 │
│  ### Applicant                                                                                                  │
│  Applicant ID: APP-1002, hereinafter referred to as Applicant B.                                                │
│                                                                                                                 │
│  ### Financial Summary                                                                                          │
│  The applicant's financial health is characterized by a debt-to-income (DTI) ratio of 16.7%, which is           │
│  considered low. Additionally, the applicant has an excellent credit score, indicating a strong credit history  │
│  and high repayment capacity. These financial metrics suggest that the applicant is well-positioned to manage   │
│  additional debt.                                                                                               │
│                                                                                                                 │
│  ### Risk Tier                                                                                                  │
│  The credit risk tier for Applicant B is assessed as Low. This determination is based on several key factors,   │
│  including a stable employment history of 7 years at their current employer, which signifies strong income      │
│  stability. Furthermore, the applicant's low DTI ratio of 16.7% and excellent credit score collectively         │
│  indicate a high repayment capacity. These factors contribute to a low credit risk profile for the applicant's  │
│  home renovation loan of $8000.                                                                                 │
│                                                                                                                 │
│  ### Compliance Check                                                                                           │
│  A thorough review of the underwriting notes for Applicant B did not reveal any red flags or compliance         │
│  issues. The file is clear to proceed without any additional regulatory or compliance concerns.                 │
│                                                                                                                 │
│  ### Final Recommendation                                                                                       │
│  Based on the financial summary, risk tier assessment, and compliance check, it is recommended that the         │
│  application for Applicant B be Approved. The applicant's strong financial position, low credit risk, and       │
│  absence of compliance issues support this decision, indicating that the applicant is a good candidate for the  │
│  $8000 home renovation loan.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short underwriting memo for applicant APP-1002 (Applicant B) combining the financial summary,    │
│  the risk tier, and the compliance finding. End with a final recommendation: Approve, Refer for manual review,  │
│  or Decline.                                                                                                    │
│  Agent: Underwriting Report Writer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- FINAL MEMO: APP-1002 ---
## Underwriting Memo for Applicant APP-1002

### Applicant
Applicant ID: APP-1002, hereinafter referred to as Applicant B.

### Financial Summary
The applicant's financial health is characterized by a debt-to-income (DTI) ratio of 16.7%, which is considered low. Additionally, the applicant has an excellent credit score, indicating a strong credit history and high repayment capacity. These financial metrics suggest that the applicant is well-positioned to manage additional debt.

### Risk Tier
The credit risk tier for Applicant B is assessed as Low. This determination is based on several key factors, including a stable employment history of 7 years at their current employer, which signifies strong income stability. Furthermore, the applicant's low DTI ratio of 16.7% and excellent credit score collectively indicate a high repayment capacity. These factors contribute to a low credit risk profile for the applicant's home renovation loan of $8000.

### Complian

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 48433539-61dd-42cd-86e7-8cf845e3708c                                                                       │
│  Final Output: ## Underwriting Memo for Applicant APP-1002                                                      │
│                                                                                                                 │
│  ### Applicant                                                                                                  │
│  Applicant ID: APP-1002, hereinafter referred to as Applicant B.                                                │
│                                                                                                                 │
│  ### Financial Summary                                                                                          │
│  The applicant's financial health is characterized by a debt-to-income (DTI) ratio of 16.7%, which is           │
│  considered low. Additionally, the applicant has an excellent credit score, indicating a strong credit history  │
│  and high repayment capacity. These financial metrics suggest that the applicant is well-positioned to manage   │
│  additional debt.                                                                                               │
│                                                                                                                 │
│  ### Risk Tier                                                                                                  │
│  The credit risk tier for Applicant B is assessed as Low. This determination is based on several key factors,   │
│  including a stable employment history of 7 years at their current employer, which signifies strong income      │
│  stability. Furthermore, the applicant's low DTI ratio of 16.7% and excellent credit score collectively         │
│  indicate a high repayment capacity. These factors contribute to a low credit risk profile for the applicant's  │
│  home renovation loan of $8000.                                                                                 │
│                                                                                                                 │
│  ### Compliance Check                                                                                           │
│  A thorough review of the underwriting notes for Applicant B did not reveal any red flags or compliance         │
│  issues. The file is clear to proceed without any additional regulatory or compliance concerns.                 │
│                                                                                                                 │
│  ### Final Recommendation                                                                                       │
│  Based on the financial summary, risk tier assessment, and compliance check, it is recommended that the         │
│  application for Applicant B be Approved. The applicant's strong financial position, low credit risk, and       │
│  absence of compliance issues support this decision, indicating that the applicant is a good candidate for the  │
│  $8000 home renovation loan.                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

## 8. Save the memos to a markdown report

In [38]:
with open("underwriting_reports.md", "w") as f:
    for app_id, output in results.items():
        f.write(f"# Underwriting Memo — {app_id}\n\n{output}\n\n---\n\n")

print("Saved: underwriting_reports.md")


Saved: underwriting_reports.md




**What this demonstrates:**
- Multi-agent orchestration with a real division of labor (not one LLM prompt pretending to be four people)
- Tool-grounded outputs — the DTI math and the KYC keyword screen are deterministic Python, not LLM guesses, which is exactly the kind of auditability a bank would ask for
- Sequential task chaining with `context=[...]` so each agent builds on verified upstream work
- Free-tier friendly: Groq's inference is fast enough that even 4 chained agents finish in seconds, and Colab needs no GPU because the LLM call is remote

